# Apache Hudi - Upserts & Duplicate Handling

The notebook creates its own Hudi **COPY_ON_WRITE** table, writes a baseline dataset, performs an upsert batch with duplicate keys, and then compares the latest snapshot with an older version using time travel.

Goal:

- create an independent Hudi table for this scenario
- write baseline data
- process updates and duplicate keys
- understand how the precombine field selects the winning record
- query the latest snapshot
- query an older table version using Hudi time travel

---


# What you will learn

Apache Hudi supports record-level updates on a data lake table by combining a stable record key with a precombine field.

Unlike plain Parquet datasets, where appending duplicate business keys creates multiple physical rows, Hudi can resolve records with the same key and keep the latest version according to the configured ordering field.

In this notebook, you will learn:

- How Hudi performs upserts into an existing Copy-On-Write table
- How duplicate record keys are resolved during writes
- Why the `recordkey.field` identifies the logical record
- Why the `precombine.field` decides which duplicate version wins
- How to inspect Hudi commit metadata columns
- How to query the latest table snapshot
- How to use Hudi time travel with `as.of.instant`
- How to compare the first commit with the latest version of the table


In [1]:
import os
import shutil
import pathlib
import datetime as dt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType


## Step 1 — Spark and Hudi configuration

The notebook is prepared for a Docker-based Spark standalone cluster.

- Spark master: `spark://spark-master:7077`
- Shared data folder: `/workspace/data`
- Hudi table path: `/workspace/data/hudi/riders_cow`

The Hudi JAR should already be available in the Docker image under `/opt/spark/jars`. Because of that, this notebook does **not** use `spark.jars.packages` and will not download Hudi dependencies at runtime.


In [2]:
MASTER = "spark://spark-master:7077"
WAREHOUSE_DIR = "/workspace/data/warehouse"
HUDI_BASE_DIR = "/workspace/data/hudi"
TABLE_NAME = "riders_cow_upserts"
TABLE_PATH = f"{HUDI_BASE_DIR}/{TABLE_NAME}"

# Keep the notebook reproducible. Every run starts from a clean demo table.
RESET_TABLE = True

pathlib.Path(HUDI_BASE_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(WAREHOUSE_DIR).mkdir(parents=True, exist_ok=True)

print(f"Spark master : {MASTER}")
print(f"Table name   : {TABLE_NAME}")
print(f"Table path   : {TABLE_PATH}")
print(f"Reset table  : {RESET_TABLE}")


Spark master : spark://spark-master:7077
Table name   : riders_cow_upserts
Table path   : /workspace/data/hudi/riders_cow_upserts
Reset table  : True


## Step 2 — Create SparkSession

Hudi needs the Kryo serializer. The Spark SQL extension and catalog make the session ready for Hudi SQL features as well.


In [3]:
spark = (
    SparkSession.builder
    .appName("Hudi-02-Upserts-Duplicate-Handling")
    .master(MASTER)
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Application : {spark.sparkContext.appName}")
print(f"Master      : {spark.sparkContext.master}")


Spark version: 4.0.2
Application : Hudi-02-Upserts-Duplicate-Handling
Master      : spark://spark-master:7077


## Step 3 — Define reusable Hudi write options

These options define how Hudi identifies records and resolves duplicates.

Key concepts:

- `recordkey.field`: stable business key for each record
- `partitionpath.field`: physical table partitioning field
- `precombine.field`: timestamp-like field used to choose the latest record when duplicate keys appear in the same batch
- `COPY_ON_WRITE`: updates are written into new Parquet files during commit


In [4]:
hudi_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.hive_style_partitioning": "true",
}

schema = StructType([
    StructField("rider", StringType(), False),
    StructField("city", StringType(), False),
    StructField("ts", TimestampType(), False),
])


## Step 4 — Create an independent baseline table

This notebook creates its own table from scratch. This avoids dependency on Notebook 01, previous kernel state, or files written to another container.

For reproducible training runs, `RESET_TABLE = True` removes the demo table first and then writes a clean baseline dataset.


In [5]:
if RESET_TABLE and os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)
    print(f"Removed existing demo table: {TABLE_PATH}")

initial_rows = [
    ("r1", "SFO", dt.datetime(2024, 1, 1, 10, 0, 0)),
    ("r2", "NYC", dt.datetime(2024, 1, 1, 11, 0, 0)),
]

initial_df = spark.createDataFrame(initial_rows, schema)

(
    initial_df.write
    .format("hudi")
    .options(**hudi_options)
    .option("hoodie.datasource.write.operation", "insert")
    .mode("overwrite")
    .save(TABLE_PATH)
)

print(f"Created independent base Hudi table: {TABLE_PATH}")


# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope


[Stage 23:>                                                         (0 + 2) / 2]

26/04/25 19:25:48 WARN HoodieTableFileSystemView: Partition: city=SFO is not available in store
26/04/25 19:25:48 WARN HoodieTableFileSystemView: Partition: city=NYC is not available in store


Created independent base Hudi table: /workspace/data/hudi/riders_cow_upserts


## Step 5 — Read the current snapshot

A snapshot query returns the latest version of each Hudi record.


In [6]:
base_df = spark.read.format("hudi").load(TABLE_PATH)
base_df.createOrReplaceTempView(TABLE_NAME)

spark.sql(f"""
    SELECT
        rider,
        city,
        ts,
        _hoodie_commit_time,
        _hoodie_record_key,
        _hoodie_partition_path
    FROM {TABLE_NAME}
    ORDER BY rider
""").show(truncate=False)


+-----+----+-------------------+-------------------+------------------+----------------------+
|rider|city|ts                 |_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|
+-----+----+-------------------+-------------------+------------------+----------------------+
|r1   |SFO |2024-01-01 10:00:00|20260425192528148  |r1                |city=SFO              |
|r2   |NYC |2024-01-01 11:00:00|20260425192528148  |r2                |city=NYC              |
+-----+----+-------------------+-------------------+------------------+----------------------+



## Step 6 — Capture the first completed commit instant

Hudi stores table changes in the `.hoodie` timeline. We will use the first completed commit later for time travel.


In [8]:
from pyspark.sql.functions import col

baseline_df = spark.read.format("hudi").load(TABLE_PATH)

baseline_df.select(
    "_hoodie_commit_time",
    "_hoodie_record_key",
    "rider",
    "city",
    "ts"
).orderBy("rider").show(truncate=False)

commits_before_update = [
    row["_hoodie_commit_time"]
    for row in baseline_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

if not commits_before_update:
    raise RuntimeError(
        f"No Hudi commit metadata found in table: {TABLE_PATH}. "
        "Check that the baseline write step completed successfully."
    )

first_commit = commits_before_update[0]
latest_commit_before_update = commits_before_update[-1]

print("Completed commits before update:")
for commit in commits_before_update:
    print(commit)

print(f"\nFirst commit instant: {first_commit}")
print(f"Latest commit before update: {latest_commit_before_update}")


+-------------------+------------------+-----+----+-------------------+
|_hoodie_commit_time|_hoodie_record_key|rider|city|ts                 |
+-------------------+------------------+-----+----+-------------------+
|20260425192528148  |r1                |r1   |SFO |2024-01-01 10:00:00|
|20260425192528148  |r2                |r2   |NYC |2024-01-01 11:00:00|
+-------------------+------------------+-----+----+-------------------+

Completed commits before update:
20260425192528148

First commit instant: 20260425192528148
Latest commit before update: 20260425192528148


## Step 7 — Prepare an update batch with duplicate keys

This batch intentionally contains multiple records for the same `rider` key.

For `r1`, Hudi should keep the row with the newest `ts` because `ts` is configured as the precombine field.


In [9]:
update_rows = [
    ("r1", "SFO", dt.datetime(2024, 1, 1, 9, 0, 0)),   # older duplicate, should lose
    ("r1", "SFO", dt.datetime(2024, 1, 2, 8, 0, 0)),   # newer duplicate, should win
    ("r2", "NYC", dt.datetime(2024, 1, 1, 10, 30, 0)),  # older than existing r2, should not win
    ("r3", "LON", dt.datetime(2024, 1, 2, 9, 0, 0)),   # new record
]

updates_df = spark.createDataFrame(update_rows, schema)

updates_df.orderBy("rider", "ts").show(truncate=False)


+-----+----+-------------------+
|rider|city|ts                 |
+-----+----+-------------------+
|r1   |SFO |2024-01-01 09:00:00|
|r1   |SFO |2024-01-02 08:00:00|
|r2   |NYC |2024-01-01 10:30:00|
|r3   |LON |2024-01-02 09:00:00|
+-----+----+-------------------+



## Step 8 — Write the update batch as an upsert

The `upsert` operation updates matching record keys and inserts new keys.

Because this is a Copy-On-Write table, Hudi rewrites affected Parquet files and exposes the new snapshot immediately after the commit.


In [10]:
(
    updates_df.write
    .format("hudi")
    .options(**hudi_options)
    .option("hoodie.datasource.write.operation", "upsert")
    .mode("append")
    .save(TABLE_PATH)
)

print("Upsert completed.")


[Stage 45:>                                                         (0 + 3) / 3]

26/04/25 19:27:35 WARN HoodieTableFileSystemView: Partition: city=LON is not available in store


26/04/25 19:27:35 WARN HoodieTableFileSystemView: Partition: city=LON is not available in store


26/04/25 19:27:39 WARN HoodieTableFileSystemView: Partition: city=LON is not available in store
Upsert completed.


## Step 9 — Query the latest snapshot

The latest snapshot should show:

- `r1` updated to `2024-01-02 08:00:00`
- `r2` unchanged because the update had an older precombine value
- `r3` inserted as a new record


In [11]:
latest_df = spark.read.format("hudi").load(TABLE_PATH)
latest_df.createOrReplaceTempView(TABLE_NAME)

spark.sql(f"""
    SELECT
        rider,
        city,
        ts,
        _hoodie_commit_time,
        _hoodie_record_key,
        _hoodie_partition_path
    FROM {TABLE_NAME}
    ORDER BY rider
""").show(truncate=False)


+-----+----+-------------------+-------------------+------------------+----------------------+
|rider|city|ts                 |_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|
+-----+----+-------------------+-------------------+------------------+----------------------+
|r1   |SFO |2024-01-02 08:00:00|20260425192730471  |r1                |city=SFO              |
|r2   |NYC |2024-01-01 11:00:00|20260425192528148  |r2                |city=NYC              |
|r3   |LON |2024-01-02 09:00:00|20260425192730471  |r3                |city=LON              |
+-----+----+-------------------+-------------------+------------------+----------------------+



## Step 10 — Verify duplicate handling and precombine behavior

These checks prove that Hudi resolved duplicates using the configured record key and precombine field.


In [12]:
actual = {
    row["rider"]: row["ts"]
    for row in spark.sql(f"SELECT rider, ts FROM {TABLE_NAME}").collect()
}

assert actual["r1"] == dt.datetime(2024, 1, 2, 8, 0, 0), "r1 should keep the newest update"
assert actual["r2"] == dt.datetime(2024, 1, 1, 11, 0, 0), "r2 should keep the original newer value"
assert actual["r3"] == dt.datetime(2024, 1, 2, 9, 0, 0), "r3 should be inserted"

row_count = spark.sql(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}").first()["c"]
assert row_count == 3, f"Expected 3 logical records, got {row_count}"

print("Validation passed.")
print(f"Rows in latest snapshot: {row_count}")
for rider, ts in sorted(actual.items()):
    print(f"{rider}: {ts}")


Validation passed.
Rows in latest snapshot: 3
r1: 2024-01-02 08:00:00
r2: 2024-01-01 11:00:00
r3: 2024-01-02 09:00:00


## Step 11 — Inspect commits after the upsert

A new commit should appear in the timeline after the upsert operation.


In [14]:
latest_df = spark.read.format("hudi").load(TABLE_PATH)

latest_df.select(
    "_hoodie_commit_time",
    "_hoodie_record_key",
    "rider",
    "city",
    "ts"
).orderBy("rider").show(truncate=False)

commits_after_update = [
    row["_hoodie_commit_time"]
    for row in latest_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

if not commits_after_update:
    raise RuntimeError(
        f"No Hudi commit metadata found in table: {TABLE_PATH}. "
        "Check that the upsert write step completed successfully."
    )

latest_commit_after_update = commits_after_update[-1]

print("Completed commits after update:")
for commit in commits_after_update:
    print(commit)

print(f"\nLatest commit after update: {latest_commit_after_update}")


+-------------------+------------------+-----+----+-------------------+
|_hoodie_commit_time|_hoodie_record_key|rider|city|ts                 |
+-------------------+------------------+-----+----+-------------------+
|20260425192730471  |r1                |r1   |SFO |2024-01-02 08:00:00|
|20260425192528148  |r2                |r2   |NYC |2024-01-01 11:00:00|
|20260425192730471  |r3                |r3   |LON |2024-01-02 09:00:00|
+-------------------+------------------+-----+----+-------------------+

Completed commits after update:
20260425192528148
20260425192730471

Latest commit after update: 20260425192730471


## Step 12 — Time travel to the first commit

Time travel allows reading the table as it existed at a specific Hudi instant.

The official Spark DataSource option for this is `as.of.instant`.


In [15]:
riders_v1_df = (
    spark.read
    .format("hudi")
    .option("as.of.instant", first_commit)
    .load(TABLE_PATH)
)

riders_v1_df.createOrReplaceTempView("riders_v1")

spark.sql("""
    SELECT
        rider,
        city,
        ts,
        _hoodie_commit_time
    FROM riders_v1
    ORDER BY rider
""").show(truncate=False)


+-----+----+-------------------+-------------------+
|rider|city|ts                 |_hoodie_commit_time|
+-----+----+-------------------+-------------------+
|r1   |SFO |2024-01-01 10:00:00|20260425192528148  |
|r2   |NYC |2024-01-01 11:00:00|20260425192528148  |
+-----+----+-------------------+-------------------+



## Step 13 — Compare first commit vs latest snapshot

This makes the effect of the upsert visible side by side.


In [16]:
first_snapshot = spark.sql("""
    SELECT
        'first_commit' AS version,
        rider,
        city,
        ts
    FROM riders_v1
""")

latest_snapshot = spark.sql(f"""
    SELECT
        'latest' AS version,
        rider,
        city,
        ts
    FROM {TABLE_NAME}
""")

(
    first_snapshot
    .unionByName(latest_snapshot)
    .orderBy("rider", "version")
    .show(truncate=False)
)


+------------+-----+----+-------------------+
|version     |rider|city|ts                 |
+------------+-----+----+-------------------+
|first_commit|r1   |SFO |2024-01-01 10:00:00|
|latest      |r1   |SFO |2024-01-02 08:00:00|
|first_commit|r2   |NYC |2024-01-01 11:00:00|
|latest      |r2   |NYC |2024-01-01 11:00:00|
|latest      |r3   |LON |2024-01-02 09:00:00|
+------------+-----+----+-------------------+



## Step 14 — Optional cleanup

Keep this cell commented out if you want to inspect the generated Hudi table after the notebook finishes.


In [ ]:
# shutil.rmtree(TABLE_PATH)
# print(f"Removed demo table: {TABLE_PATH}")
